# Jugular vein ultrasound: subject-grouped segmentation and area measurement (GPU)

Implements the *Jugular Vein Model Plan*: SegThy audit with a slice-axis check, transverse slices at a fixed 0.25 mm per pixel, a four-level U-Net (background, thyroid, carotid, jugular vein), 4-fold cross-validation by subject, and rule-based area measurements.

**Data:** SegThy manual ultrasound annotations, 31 usable volumes from 16 healthy volunteers, through a pinned, checksum-verified MedOtter mirror (third-party redistribution). CC BY 4.0; cite Kroenke et al., PLOS ONE 2022, doi:10.1371/journal.pone.0268550.

**Scope:** outlines and image-plane areas only. No blood flow, clots, venous pressure or diagnosis. SegThy slices are cut from reconstructed 3D volumes and are not ordinary probe frames. Not validated for astronauts or clinical use.

**Before running:** attach a GPU. On Kaggle: Settings > Accelerator > GPU, and turn Internet on (the dataset is downloaded). On Colab: Runtime > Change runtime type > GPU. The settings cell stops if no GPU is attached.

In [ ]:
EPOCHS = 60          # plan's starting value; the best epoch per fold is kept using 2 training-side subjects
USE_DRIVE = False    # Colab only: True keeps data, checkpoints and results on Google Drive

import os, torch
assert torch.cuda.is_available(), "No GPU attached. Turn on a GPU accelerator for this notebook, then run again."
print("GPU:", torch.cuda.get_device_name(0))
ON_KAGGLE = os.path.exists("/kaggle/working")
root = "/kaggle/working/jugular_work" if ON_KAGGLE else "/content/jugular_work"
if USE_DRIVE and not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
    root = "/content/drive/MyDrive/jugular_work"
os.environ.update(JUGULAR_ROOT=root, JUGULAR_EPOCHS=str(EPOCHS), JUGULAR_TRAIN_STRIDE="1", JUGULAR_RUN="run_gpu")
os.makedirs(root, exist_ok=True)
print("Working folder:", root)
get_ipython().run_line_magic("pip", "-q install nibabel")

## Source files
The next three cells write the pipeline modules to disk. Nothing is trained here.

In [ ]:
%%writefile jugular_data.py
"""Jugular plan steps 1-3: download SegThy, audit it, cut transverse slices at a fixed physical scale.

Research use only. SegThy is CC BY 4.0 (Kroenke et al., PLOS ONE 2022, doi:10.1371/journal.pone.0268550).
Data comes from the pinned, checksum-verified MedOtter mirror (third-party redistribution) of the
manual ultrasound annotations. Raw data is kept outside the git repository (see JUGULAR_ROOT).

Labels in the source masks: 0 background, 1 thyroid, 2 left carotid, 3 left jugular, 4 right carotid,
5 right jugular. Here they become 0 background, 1 thyroid, 2 carotid, 3 jugular vein; the neck side is
kept as metadata.
"""
import hashlib
import json
import os
import sys
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

ROOT = Path(os.environ.get("JUGULAR_ROOT") or ("/kaggle/working/jugular_work" if Path("/kaggle/working").exists() else "/content/jugular_work" if Path("/content").exists() else r"C:\Users\nikhi\Downloads\jugular_work"))
RAW = ROOT / "raw"
PROC = ROOT / "processed"
REV = "318d0e04578419d2976a093f9873f14b782f253d"
HF = f"https://huggingface.co/datasets/MedOtter/SegThy/resolve/{REV}/"
SPACING = 0.25          # mm per pixel after resampling
STRIDE = 4              # keep every 4th transverse plane of the 0.12 mm volume (about 0.5 mm apart)
AXIS = 2                # transverse (axial) axis; checked by audit() against the header and the vessel shapes
REMAP = np.array([0, 1, 2, 3, 2, 3], dtype="uint8")
CLASSES = ("background", "thyroid", "carotid", "vein")
THYROID, CAROTID, VEIN = 1, 2, 3


def _get(url, tries=5, stream=False):
    import requests

    for k in range(tries):
        try:
            r = requests.get(url, timeout=180, stream=stream)
            r.raise_for_status()
            return r
        except Exception:
            if k == tries - 1:
                raise
            time.sleep(3 * (k + 1))


def _sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _fetch(rel, sha):
    path = RAW / rel
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix(path.suffix + ".part")
        with _get(HF + rel, stream=True) as r, open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        tmp.replace(path)
    if _sha(path) != sha:
        path.unlink()
        raise ValueError(f"checksum mismatch, file removed: {rel}")
    return path


def download():
    """Fetch the metadata and every manually annotated ultrasound volume, verifying SHA-256."""
    RAW.mkdir(parents=True, exist_ok=True)
    meta_path = RAW / "us_train.jsonl"
    if not meta_path.exists():
        meta_path.write_bytes(_get(HF + "us_train.jsonl").content)
    meta = [json.loads(x) for x in meta_path.read_text().splitlines() if x.strip()]
    jobs = [(r[k], r[k + "_sha256"]) for r in meta for k in ("image", "mask")]
    with ThreadPoolExecutor(max_workers=4) as pool:
        for i, _ in enumerate(pool.map(lambda j: _fetch(*j), jobs)):
            if i % 8 == 0:
                print("downloaded files:", i + 1, "of", len(jobs), flush=True)
    print("volumes:", len(meta), "subjects:", len({r["subject_id"] for r in meta}))
    return meta


def _elongation(mask2d):
    """Long axis / short axis of a binary region from its second moments (1 = round)."""
    ys, xs = np.nonzero(mask2d)
    if len(ys) < 20:
        return None
    c = np.cov(np.vstack([ys, xs]))
    e = np.sort(np.linalg.eigvalsh(c))
    return float(np.sqrt(e[1] / max(e[0], 1e-9)))


def audit():
    """Check files, labels and alignment, and confirm which axis gives transverse vessel cross-sections.

    A vessel cut across its length is a compact round or oval region; cut along its length it is a long
    strip. For every axis the median elongation of the jugular and carotid regions is measured; the
    transverse axis must be the one with the roundest cross-sections and must match the NIfTI
    orientation codes (the axis labelled S or I, since neck vessels run head to foot).
    """
    import nibabel as nib

    meta = [json.loads(x) for x in (RAW / "us_train.jsonl").read_text().splitlines() if x.strip()]
    problems = defaultdict(list)
    elong = {a: [] for a in range(3)}
    header_axes = []
    volumes = []
    for r in meta:
        ni, nm = nib.load(str(RAW / r["image"])), nib.load(str(RAW / r["mask"]))
        y = np.asarray(nm.dataobj)
        if ni.shape != nm.shape or y.ndim != 3:
            problems["shape_mismatch"].append(r["case_id"]); continue
        if not np.allclose(ni.affine, nm.affine, atol=1e-3):
            problems["affine_mismatch"].append(r["case_id"])
        values = sorted(int(v) for v in np.unique(y))
        if not set(values) <= {0, 1, 2, 3, 4, 5}:
            problems["unexpected_labels"].append(r["case_id"]); continue
        side_ok = set(values) <= ({0, 1, 2, 3} if r["side"] == "left" else {0, 1, 4, 5})
        if not side_ok:
            problems["labels_do_not_match_side"].append(r["case_id"])
        has_vein = bool(np.isin(y, [3, 5]).any())
        if r.get("annotation_gap") or not has_vein:
            problems["vein_not_annotated_excluded"].append(r["case_id"])
        codes = nib.aff2axcodes(ni.affine)
        header_axes.append([i for i, c in enumerate(codes) if c in "SI"][0])
        zooms = [float(z) for z in ni.header.get_zooms()[:3]]
        if has_vein:
            vessel = np.isin(y, [2, 3, 4, 5])
            for a in range(3):
                idx = np.nonzero(vessel.any(axis=tuple(k for k in range(3) if k != a)))[0]
                for z in idx[:: max(len(idx) // 12, 1)]:
                    sl = np.take(y, z, axis=a)
                    for lab in ((2, 4), (3, 5)):
                        e = _elongation(np.isin(sl, lab))
                        if e is not None:
                            elong[a].append(e)
        volumes.append(dict(case=r["case_id"], subject=r["subject_id"], side=r["side"], shape=list(y.shape), spacing_mm=zooms,
                            axcodes="".join(codes), labels=values, usable=bool(has_vein and not r.get("annotation_gap"))))
        print("audited", r["case_id"], y.shape, zooms, "".join(codes), values, flush=True)
    med = {a: float(np.median(v)) for a, v in elong.items() if v}
    roundest = min(med, key=med.get)
    out = dict(
        source=HF, license="CC BY 4.0; cite Kroenke et al., PLOS ONE 2022; MedOtter mirror is a third-party redistribution",
        volumes=len(volumes), usable_volumes=sum(v["usable"] for v in volumes), subjects=len({v["subject"] for v in volumes}),
        usable_subjects=len({v["subject"] for v in volumes if v["usable"]}),
        median_vessel_elongation_by_axis=med, roundest_axis=roundest, header_head_foot_axis=sorted(set(header_axes)),
        transverse_axis_confirmed=bool(roundest == AXIS and set(header_axes) == {AXIS}),
        problems=dict(problems), per_volume=volumes)
    (ROOT / "audit.json").write_text(json.dumps(out, indent=1))
    print(json.dumps({k: v for k, v in out.items() if k != "per_volume"}, indent=2))
    assert out["transverse_axis_confirmed"], "axis check failed: do not train until this is resolved"
    return out


def resample(a, shape, nearest):
    from PIL import Image

    return np.asarray(Image.fromarray(a).resize((shape[1], shape[0]), Image.Resampling.NEAREST if nearest else Image.Resampling.BILINEAR))


def preprocess():
    """Write one file per volume: every STRIDE-th transverse slice at SPACING mm per pixel.

    Each slice is transposed and flipped so rows run from the skin (top, as on a scanner screen) to
    deep tissue and columns run across the neck. In these volumes the skin is at the high end of
    axis 1 in all 31 scans (checked from brightness and vessel depth), whatever the header code says.
    Images stay uint8; brightness scaling happens at load time. Reference
    areas are counted on the ORIGINAL 0.12 mm mask so resampling error is part of the measured error.
    """
    import nibabel as nib

    audit_ = json.loads((ROOT / "audit.json").read_text())
    assert audit_["transverse_axis_confirmed"]
    PROC.mkdir(exist_ok=True)
    manifest = []
    for v in audit_["per_volume"]:
        if not v["usable"]:
            continue
        ni, nm = nib.load(str(RAW / f"us/train/images/{v['case']}.nii.gz")), nib.load(str(RAW / f"us/train/masks/{v['case']}.nii.gz"))
        x, y = np.asarray(ni.dataobj), REMAP[np.asarray(nm.dataobj)]
        assert x.dtype == np.uint8, x.dtype
        sp = v["spacing_mm"]
        assert abs(sp[0] - sp[1]) < 1e-6
        shape = (int(round(x.shape[1] * sp[1] / SPACING)), int(round(x.shape[0] * sp[0] / SPACING)))
        imgs, masks, zs, ref = [], [], [], []
        for z in range(0, x.shape[AXIS], STRIDE):
            a, b = x[:, :, z].T[::-1], y[:, :, z].T[::-1]
            imgs.append(resample(np.ascontiguousarray(a), shape, False))
            masks.append(resample(np.ascontiguousarray(b), shape, True))
            zs.append(z)
            ref.append([float((b == k).sum() * sp[0] * sp[1]) for k in (1, 2, 3)])
        out = PROC / f"{v['case']}.npz"
        np.savez_compressed(out, image=np.stack(imgs), mask=np.stack(masks), z=np.array(zs), ref_area_mm2=np.array(ref, dtype="float32"))
        m = np.stack(masks)
        manifest.append(dict(case=v["case"], subject=v["subject"], side=v["side"], path=str(out), slices=len(zs), shape=list(shape),
                             spacing_mm=SPACING, source_spacing_mm=sp, slice_gap_mm=sp[2] * STRIDE,
                             vein_slices=int((m == VEIN).any(axis=(1, 2)).sum()), carotid_slices=int((m == CAROTID).any(axis=(1, 2)).sum())))
        print("preprocessed", v["case"], shape, "slices", len(zs), "with vein", manifest[-1]["vein_slices"], flush=True)
    (ROOT / "manifest.json").write_text(json.dumps(manifest, indent=1))
    subjects = sorted({r["subject"] for r in manifest})
    print(json.dumps(dict(volumes=len(manifest), subjects=len(subjects), slices=sum(r["slices"] for r in manifest),
                          vein_slices=sum(r["vein_slices"] for r in manifest))))
    return manifest


if __name__ == "__main__":
    ROOT.mkdir(parents=True, exist_ok=True)
    if "download" in sys.argv:
        download()
    if "audit" in sys.argv:
        audit()
    if "preprocess" in sys.argv:
        preprocess()

In [ ]:
%%writefile jugular_geometry.py
"""Rule-based measurements from a jugular segmentation mask, plus baseline tracking.

Nothing here is learned. Every reported number can be traced to the outline it came from.
These are geometric measurements only: they say nothing about blood flow, clots, venous pressure
or pressure inside the skull. Not validated on astronauts or for clinical use.
"""
import numpy as np
from scipy import ndimage

from jugular_data import CAROTID, VEIN

MIN_VEIN_MM2 = 3.0        # smaller vein regions are reported as "vein not found" (starting value)
MIN_VEIN_PX = 40          # same idea when no pixel spacing is available
FLAT_LIMIT = 0.25         # short/long axis below this suggests probe pressure (starting value)
UNSTABLE_LIMIT = 0.15     # relative area difference between the frame and its mirror image (starting value)
FLAG_SPREADS = 2.0        # a change is flagged beyond this many baseline spreads...
FLAG_SESSIONS = 2         # ...on this many sessions in a row
NO_FLOW_NOTE = "This measurement does not show blood flow or clots."


def largest_region(binary):
    """Largest connected region with holes filled; None if empty."""
    lab, n = ndimage.label(binary)
    if n == 0:
        return None
    sizes = ndimage.sum(binary, lab, index=np.arange(1, n + 1))
    return ndimage.binary_fill_holes(lab == 1 + int(np.argmax(sizes)))


def ellipse_axes(region):
    """Full long and short axis (pixels) of the ellipse with the same second moments as the region."""
    ys, xs = np.nonzero(region)
    if len(ys) < 3:
        return 0.0, 0.0
    e = np.sort(np.linalg.eigvalsh(np.cov(np.vstack([ys, xs])) + np.eye(2) / 12.0))
    return float(4 * np.sqrt(e[1])), float(4 * np.sqrt(e[0]))


def measure_frame(mask, spacing_mm=None):
    """Measurements for one predicted mask. spacing_mm=None gives only the scale-free outputs."""
    vein, artery = largest_region(mask == VEIN), largest_region(mask == CAROTID)
    px = 0 if vein is None else int(vein.sum())
    sq = None if spacing_mm is None else float(spacing_mm) ** 2
    found = px >= MIN_VEIN_PX if sq is None else px * sq >= MIN_VEIN_MM2
    out = dict(vein_found=bool(found), area_mm2=None, long_axis_mm=None, short_axis_mm=None, flattening=None,
               vein_to_artery=None, artery_area_mm2=None, vein_pixels=px)
    if not found:
        return out
    long_px, short_px = ellipse_axes(vein)
    out["flattening"] = short_px / long_px if long_px else None
    apx = 0 if artery is None else int(artery.sum())
    if apx >= MIN_VEIN_PX:
        out["vein_to_artery"] = px / apx
    if sq is not None:
        out.update(area_mm2=px * sq, long_axis_mm=long_px * spacing_mm, short_axis_mm=short_px * spacing_mm,
                   artery_area_mm2=apx * sq if apx else None)
    return out


def quality(m, m_mirror=None):
    """Accept or rescan, with reasons. m_mirror is the measurement from the mirrored frame's prediction."""
    reasons = []
    if not m["vein_found"]:
        reasons.append("vein not found")
    else:
        if m["flattening"] is not None and m["flattening"] < FLAT_LIMIT:
            reasons.append("vein looks flattened: ease probe pressure")
        if m_mirror is not None:
            a, b = m["vein_pixels"], m_mirror["vein_pixels"]
            if abs(a - b) / max((a + b) / 2, 1) > UNSTABLE_LIMIT:
                reasons.append("prediction unstable")
    return dict(accept=not reasons, reasons=reasons)


def measure_clip(masks, spacing_mm):
    """Area trace over a cine clip with a 3-frame median filter, and (max - min) / mean. Experimental."""
    areas = np.array([(measure_frame(m, spacing_mm)["area_mm2"] or np.nan) for m in masks], dtype=float)
    if np.isnan(areas).mean() > 0.2 or len(areas) < 3:
        return dict(trace_mm2=areas.tolist(), pulsatility=None, mean_area_mm2=None, note="vein not found in enough frames")
    filled = np.where(np.isnan(areas), np.nanmedian(areas), areas)
    trace = ndimage.median_filter(filled, size=3, mode="nearest")
    return dict(trace_mm2=trace.tolist(), mean_area_mm2=float(trace.mean()), max_area_mm2=float(trace.max()), min_area_mm2=float(trace.min()),
                pulsatility=float((trace.max() - trace.min()) / trace.mean()), note="experimental: frame-by-frame by-product, not validated")


def make_baseline(areas_mm2):
    """Baseline from at least three preflight scans of one side in one posture (supine recommended)."""
    a = np.asarray(areas_mm2, dtype=float)
    if len(a) < 3:
        raise ValueError("a baseline needs at least three scans")
    return dict(median_mm2=float(np.median(a)), spread_mm2=float(np.std(a, ddof=1)), scans=int(len(a)))


def compare(area_mm2, baseline, earlier_outside=0):
    """Change from the person's own baseline. earlier_outside = consecutive earlier sessions already outside range."""
    diff = area_mm2 - baseline["median_mm2"]
    outside = abs(diff) > FLAG_SPREADS * baseline["spread_mm2"]
    run = earlier_outside + 1 if outside else 0
    return dict(percent_change=100.0 * diff / baseline["median_mm2"], outside_own_range=bool(outside),
                sessions_outside_in_a_row=run, flagged=bool(run >= FLAG_SESSIONS))


def report_line(side, m, q, change=None):
    """The single line shown to the crew member."""
    if not q["accept"]:
        return f"{side.capitalize()} jugular: rescan needed ({'; '.join(q['reasons'])})."
    if m["area_mm2"] is None:
        ratio = "not available" if m["vein_to_artery"] is None else f"{m['vein_to_artery']:.2f}"
        return f"{side.capitalize()} jugular: no pixel spacing, so no area. Vein-to-artery ratio {ratio}. {NO_FLOW_NOTE}"
    text = f"{side.capitalize()} jugular area {m['area_mm2']:.0f} mm2"
    if change is not None:
        word = "above" if change["percent_change"] >= 0 else "below"
        text += f", {abs(change['percent_change']):.0f}% {word} your supine baseline"
        text += ", outside your normal range" if change["outside_own_range"] else ", within your normal range"
    text += f". {NO_FLOW_NOTE}"
    if change is not None and change["flagged"]:
        text += " Consider running the full Doppler exam and informing the flight surgeon."
    return text

In [ ]:
%%writefile jugular_pipeline.py
"""Jugular plan steps 4-6: subject-grouped cross-validation of a small U-Net, then measurement checks.

Stages (cached on disk, resumable at every epoch):
  cv     : 4 folds by subject. Each fold trains on 12 subjects (2 of them choose the checkpoint)
           and predicts the 4 held-out subjects exactly once.
  report : per-subject Dice, area error against the manual outline, artery/vein confusion, speed,
           and the plan's acceptance targets.

Rules enforced in code:
  * both neck sides of a subject are always in the same fold
  * checkpoint-selection subjects come from inside the training subjects, never the held-out fold
  * held-out predictions use images only; manual masks are read only to score them
Research baseline on 16 healthy volunteers. No astronaut, flow, clot or clinical validation.
"""
import argparse
import hashlib
import json
import math
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

from jugular_data import CAROTID, CLASSES, ROOT, SPACING, THYROID, VEIN
from jugular_geometry import largest_region, measure_frame, quality

F = nn.functional
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = bool(os.environ.get("JUGULAR_SMOKE"))
RUN = ROOT / ("run_smoke" if SMOKE else os.environ.get("JUGULAR_RUN", "run_gpu" if DEV.type == "cuda" else "run_local"))
FITS = RUN / "fits"
SEED = 42
CFG = dict(
    model="unet24-48-96-192-gn", classes=len(CLASSES), crop=224, spacing_mm=SPACING,
    epochs=1 if SMOKE else int(os.environ.get("JUGULAR_EPOCHS", 60)),
    # train_stride 1 uses every preprocessed slice (0.5 mm apart); 2 uses every other one (CPU budget)
    train_stride=int(os.environ.get("JUGULAR_TRAIN_STRIDE", 1)),
    batch=12, lr=3e-4, weight_decay=1e-4, opt="adamw+cosine", loss="ce+softdice", folds=4, stop_subjects=2, empty_cap=0.20,
    aug=dict(flip=0.5, rot_deg=20, scale=(0.7, 1.5), shift=0.08, elastic=0.03, gamma=0.25, contrast=0.25, brightness=0.10,
             speckle=0.15, blur_p=0.25, wedge_p=0.30, wedge_width=0.25),
    seed=SEED)
TARGETS = dict(mean_vein_dice=0.85, min_subject_vein_dice=0.70, median_area_error=0.10, area_bias=0.05, artery_as_vein=0.02, seconds_per_frame=1.0)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def hms(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}:{sec % 3600 // 60:02d}:{sec % 60:02d}"


# ----------------------------------------------------------------------------- data
class Data:
    def __init__(self):
        self.manifest = json.loads((ROOT / "manifest.json").read_text())
        self.vols = {}
        for r in self.manifest:
            with np.load(r["path"]) as d:
                img, mask = d["image"], d["mask"]
                lo, hi = np.percentile(img.reshape(len(img), -1), [1, 99], axis=1)
                self.vols[r["case"]] = dict(image=img, mask=mask, ref=d["ref_area_mm2"], z=d["z"], lo=lo.astype("float32"),
                                            hi=hi.astype("float32"), has_vein=(mask == VEIN).any(axis=(1, 2)), subject=r["subject"], side=r["side"])
        self.subjects = sorted({r["subject"] for r in self.manifest})
        self.cases = {s: [r["case"] for r in self.manifest if r["subject"] == s] for s in self.subjects}

    def image(self, case, i):
        v = self.vols[case]
        return np.clip((v["image"][i].astype("float32") - v["lo"][i]) / max(float(v["hi"][i] - v["lo"][i]), 1e-6), 0, 1)

    def samples(self, subjects, stride=1):
        vein, empty = [], []
        for s in subjects:
            for c in self.cases[s]:
                hv = self.vols[c]["has_vein"]
                for i in range(0, len(hv), stride):
                    (vein if hv[i] else empty).append((c, i))
        return vein, empty


def normalise(a):
    lo, hi = np.percentile(a, [1, 99])
    return np.clip((a.astype("float32") - lo) / max(float(hi - lo), 1e-6), 0, 1)


def make_folds(subjects):
    s = list(subjects)
    random.Random(SEED).shuffle(s)
    n = CFG["folds"]
    folds = []
    for k in range(n):
        held = sorted(s[k::n])
        rest = [x for x in s if x not in held]
        stop = sorted(rest[-CFG["stop_subjects"]:]) if k % 2 == 0 else sorted(rest[:CFG["stop_subjects"]])
        train = sorted(x for x in rest if x not in stop)
        assert not set(held) & set(train) and not set(held) & set(stop) and not set(train) & set(stop)
        folds.append(dict(fold=k, train=train, stop=stop, held_out=held))
    assert sorted(x for f in folds for x in f["held_out"]) == sorted(s), "every subject is held out exactly once"
    return folds


# ----------------------------------------------------------------------------- model
def block(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.GroupNorm(8, b), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.GroupNorm(8, b), nn.ReLU())


class UNet(nn.Module):
    """Four resolution levels, 24/48/96/192 channels, about 1.1 million parameters."""

    def __init__(self, n=len(CLASSES)):
        super().__init__()
        self.a = block(1, 24); self.b = block(24, 48); self.c = block(48, 96); self.d = block(96, 192)
        self.e = block(288, 96); self.f = block(144, 48); self.g = block(72, 24); self.head = nn.Conv2d(24, n, 1)

    def forward(self, x):
        a = self.a(x); b = self.b(F.max_pool2d(a, 2)); c = self.c(F.max_pool2d(b, 2)); d = self.d(F.max_pool2d(c, 2))
        up = lambda t, ref: F.interpolate(t, size=ref.shape[-2:], mode="bilinear", align_corners=False)
        e = self.e(torch.cat([up(d, c), c], 1)); f = self.f(torch.cat([up(e, b), b], 1)); g = self.g(torch.cat([up(f, a), a], 1))
        return self.head(g)


def seg_loss(logits, y):
    n = logits.shape[1]
    p = logits.float().softmax(1)
    t = F.one_hot(y, n).permute(0, 3, 1, 2).float()
    dice = (2 * (p * t).sum((0, 2, 3)) + 1e-5) / (p.sum((0, 2, 3)) + t.sum((0, 2, 3)) + 1e-5)
    return F.cross_entropy(logits.float(), y) + 1 - dice[1:].mean()


def augment(x, y, gen):
    """x: B,1,H,W in 0..1; y: B,H,W. Geometry is applied to image and mask together, brightness to the image only."""
    A = CFG["aug"]
    B, _, H, W = x.shape
    r = lambda *shape: torch.rand(*shape, generator=gen, device=x.device)
    ang = (r(B) * 2 - 1) * math.radians(A["rot_deg"])
    lo, hi = A["scale"]
    zoom = torch.exp(math.log(lo) + r(B) * (math.log(hi) - math.log(lo)))
    flip = torch.where(r(B) < A["flip"], -1.0, 1.0)
    cos, sin = torch.cos(ang) / zoom, torch.sin(ang) / zoom
    theta = torch.stack([torch.stack([cos * flip, -sin, (r(B) * 2 - 1) * A["shift"]], 1), torch.stack([sin * flip, cos, (r(B) * 2 - 1) * A["shift"]], 1)], 1)
    grid = F.affine_grid(theta, (B, 1, H, W), align_corners=False)
    disp = (torch.rand(B, 2, 7, 7, generator=gen, device=x.device) * 2 - 1) * A["elastic"]
    grid = grid + F.interpolate(disp, size=(H, W), mode="bicubic", align_corners=False).permute(0, 2, 3, 1)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    y = F.grid_sample(y[:, None].float(), grid, mode="nearest", padding_mode="zeros", align_corners=False)[:, 0].long()
    v = lambda: r(B)[:, None, None, None]
    x = x.clamp(0, 1) ** torch.exp((v() * 2 - 1) * A["gamma"])
    x = (x - 0.5) * (1 + (v() * 2 - 1) * A["contrast"]) + 0.5 + (v() * 2 - 1) * A["brightness"]
    speckle = 1 + A["speckle"] * torch.randn(x.shape, generator=gen, device=x.device)
    x = torch.where(v() < 0.5, x * speckle, x)
    x = torch.where(v() < A["blur_p"], F.avg_pool2d(x, 3, stride=1, padding=1), x)
    # dark wedge: a band of columns loses brightness below a random depth (acoustic shadow / lost skin contact)
    cols = torch.arange(W, device=x.device)[None, None, None, :] / W
    rows = torch.arange(H, device=x.device)[None, None, :, None] / H
    c0, width, depth = v(), v() * A["wedge_width"], v() * 0.6
    shadow = (cols >= c0) & (cols < c0 + width) & (rows >= depth) & (v() < A["wedge_p"])
    x = torch.where(shadow, x * (0.1 + 0.4 * v()), x)
    return x.clamp(0, 1), y


def crop_batch(D, items, rng, size):
    """Random crop or zero-pad every slice to size x size."""
    X = np.zeros((len(items), 1, size, size), "float32"); Y = np.zeros((len(items), size, size), "int64")
    for k, (c, i) in enumerate(items):
        a, b = D.image(c, i), D.vols[c]["mask"][i]
        h, w = a.shape
        sy, sx = (int(rng.integers(0, h - size + 1)) if h > size else 0), (int(rng.integers(0, w - size + 1)) if w > size else 0)
        dy, dx = (int(rng.integers(0, size - h + 1)) if h < size else 0), (int(rng.integers(0, size - w + 1)) if w < size else 0)
        ch, cw = min(h, size), min(w, size)
        X[k, 0, dy:dy + ch, dx:dx + cw] = a[sy:sy + ch, sx:sx + cw]
        Y[k, dy:dy + ch, dx:dx + cw] = b[sy:sy + ch, sx:sx + cw]
    return torch.from_numpy(X), torch.from_numpy(Y)


@torch.no_grad()
def predict_probs(models, a, tta=True):
    """a: 2D float image in 0..1 at SPACING mm per pixel. Returns (probabilities, probabilities from the mirrored frame)."""
    h, w = a.shape
    H, W = math.ceil(h / 8) * 8, math.ceil(w / 8) * 8
    x = torch.zeros(1, 1, H, W, device=DEV)
    x[0, 0, :h, :w] = torch.from_numpy(np.ascontiguousarray(a)).to(DEV)
    plain = sum(m(x).float().softmax(1) for m in models) / len(models)
    mirror = None
    if tta:
        mirror = sum(m(x.flip(-1)).float().softmax(1) for m in models).flip(-1) / len(models)
    cut = lambda p: p[0, :, :h, :w].cpu().numpy()
    return cut(plain), (cut(mirror) if tta else None)


def counts(pred, ref):
    return np.array([[((pred == k) & (ref == k)).sum(), (pred == k).sum(), (ref == k).sum()] for k in (THYROID, CAROTID, VEIN)], dtype=np.int64)


def dice_of(c):
    return [float(2 * i / (p + t)) if p + t else None for i, p, t in c]


@torch.no_grad()
def stop_score(model, D, subjects):
    """Mean over the checkpoint-selection subjects of pooled vein Dice (every other slice, no mirroring)."""
    model.eval()
    scores = []
    for s in subjects:
        tot = np.zeros((3, 3), np.int64)
        for c in D.cases[s]:
            for i in range(0, len(D.vols[c]["mask"]), 2):
                p, _ = predict_probs([model], D.image(c, i), tta=False)
                tot += counts(p.argmax(0), D.vols[c]["mask"][i])
        scores.append(dice_of(tot)[2] or 0.0)
    return float(np.mean(scores))


def fit_fold(D, f):
    key = hashlib.sha256(json.dumps([CFG, f], sort_keys=True).encode()).hexdigest()[:12]
    final, last = FITS / f"fold{f['fold']}_{key}.pt", FITS / f"fold{f['fold']}_{key}.last.pt"
    if final.exists():
        log(f"fold {f['fold']}: already trained")
        return final
    FITS.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(SEED + f["fold"])
    rng = np.random.default_rng(SEED + f["fold"])
    gen = torch.Generator(device=DEV).manual_seed(SEED + f["fold"])
    model = UNet().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG["epochs"])
    scaler = torch.amp.GradScaler(enabled=DEV.type == "cuda")
    vein, empty = D.samples(f["train"], CFG["train_stride"])
    n_empty = min(len(empty), int(len(vein) * CFG["empty_cap"] / (1 - CFG["empty_cap"])))
    start, best, hist = 0, -1.0, []
    if last.exists():
        ck = torch.load(last, map_location=DEV, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        start, best, hist = ck["epoch"], ck["best"], ck["hist"]
        rng = np.random.default_rng(SEED + f["fold"] + 1000 * start)
        log(f"fold {f['fold']}: resuming after epoch {start}")
    log(f"fold {f['fold']}: train {f['train']} stop {f['stop']} held out {f['held_out']} | {len(vein)} vein slices + {n_empty} empty per epoch on {DEV}")
    t0 = time.time()
    for epoch in range(start, CFG["epochs"]):
        model.train()
        items = vein + [empty[j] for j in rng.choice(len(empty), n_empty, replace=False)]
        order = rng.permutation(len(items))
        if SMOKE:
            order = order[:48]
        total = 0.0
        for b in range(0, len(order), CFG["batch"]):
            x, y = crop_batch(D, [items[j] for j in order[b:b + CFG["batch"]]], rng, CFG["crop"])
            x, y = augment(x.to(DEV), y.to(DEV), gen)
            opt.zero_grad()
            with torch.autocast(DEV.type, enabled=DEV.type == "cuda"):
                loss = seg_loss(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            total += float(loss) * len(x)
        sched.step()
        score = stop_score(model, D, f["stop"])
        hist.append(dict(epoch=epoch + 1, loss=total / len(order), stop_vein_dice=score))
        if score > best:
            best = score
            torch.save(dict(model=model.state_dict(), cfg=CFG, fold=f, epoch=epoch + 1, stop_vein_dice=score), final.with_suffix(".best.pt"))
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), epoch=epoch + 1, best=best, hist=hist), last)
        done = epoch + 1 - start
        log(f"fold {f['fold']} epoch {epoch + 1}/{CFG['epochs']} loss {total / len(order):.4f} stop vein Dice {score:.4f} (best {best:.4f}) | "
            f"about {hms((time.time() - t0) / done * (CFG['epochs'] - epoch - 1))} left in this fold")
    final.with_suffix(".best.pt").replace(final)
    (FITS / f"fold{f['fold']}_{key}.history.json").write_text(json.dumps(hist, indent=1))
    last.unlink(missing_ok=True)
    return final


def load_model(path):
    m = UNet().to(DEV)
    m.load_state_dict(torch.load(path, map_location=DEV, weights_only=False)["model"])
    return m.eval()


def score_fold(D, f, path):
    """Predict the held-out subjects once; keep pooled Dice counts per subject and per-slice areas."""
    out = RUN / f"oof_fold{f['fold']}.json"
    if out.exists():
        return json.loads(out.read_text())
    model = load_model(path)
    subjects, slices = {}, []
    px = SPACING ** 2
    for s in f["held_out"]:
        tot = np.zeros((3, 3), np.int64)
        for c in D.cases[s]:
            v = D.vols[c]
            for i in range(len(v["mask"])):
                p, pm = predict_probs([model], D.image(c, i))
                pred, ref = ((p + pm) / 2).argmax(0), v["mask"][i]
                tot += counts(pred, ref)
                m, mm = measure_frame(pred, SPACING), measure_frame(pm.argmax(0), SPACING)
                slices.append(dict(subject=s, case=c, z=int(v["z"][i]), ref_vein_mm2=float(v["ref"][i][2]), ref_carotid_mm2=float(v["ref"][i][1]),
                                   pred_vein_mm2=m["area_mm2"] or 0.0, pred_vein_raw_mm2=float((pred == VEIN).sum() * px),
                                   vein_on_ref_carotid_px=int(((pred == VEIN) & (ref == CAROTID)).sum()), pred_vein_px=int((pred == VEIN).sum()),
                                   flattening=m["flattening"], accept=quality(m, mm)["accept"]))
        d = dice_of(tot)
        subjects[s] = dict(thyroid_dice=d[0], carotid_dice=d[1], vein_dice=d[2], fold=f["fold"])
        log(f"held-out subject {s}: vein Dice {d[2]:.4f} carotid {d[1]:.4f} thyroid {d[0]:.4f}")
    res = dict(fold=f, model=path.name, subjects=subjects, slices=slices)
    out.write_text(json.dumps(res))
    return res


def stage_cv(only=None):
    D = Data()
    folds = make_folds(D.subjects)
    RUN.mkdir(parents=True, exist_ok=True)
    (RUN / "folds.json").write_text(json.dumps(folds, indent=1))
    (RUN / "run_info.json").write_text(json.dumps(dict(cfg=CFG, device=str(DEV), torch=torch.__version__, platform=platform.platform(),
                                                        volumes=len(D.manifest), subjects=D.subjects), indent=1))
    for f in folds:
        if only is None or f["fold"] in only:
            score_fold(D, f, fit_fold(D, f))


def stage_report():
    folds = json.loads((RUN / "folds.json").read_text())
    done = [json.loads(p.read_text()) for p in sorted(RUN.glob("oof_fold*.json"))]
    subjects = {s: v for r in done for s, v in r["subjects"].items()}
    rows = [x for r in done for x in r["slices"]]
    per = {}
    for s in sorted(subjects):
        mine = [x for x in rows if x["subject"] == s]
        big = [x for x in mine if x["ref_vein_mm2"] > 20]
        err = [abs(x["pred_vein_mm2"] - x["ref_vein_mm2"]) / x["ref_vein_mm2"] for x in big]
        present = [x for x in mine if x["ref_vein_mm2"] >= 3]
        absent = [x for x in mine if x["ref_vein_mm2"] == 0]
        per[s] = dict(**subjects[s], slices=len(mine),
                      median_area_error=float(np.median(err)) if err else None,
                      area_bias=float((sum(x["pred_vein_mm2"] for x in big) - sum(x["ref_vein_mm2"] for x in big)) / sum(x["ref_vein_mm2"] for x in big)) if big else None,
                      artery_as_vein=float(sum(x["vein_on_ref_carotid_px"] for x in mine) / max(sum(x["pred_vein_px"] for x in mine), 1)),
                      vein_found_when_present=float(np.mean([x["pred_vein_mm2"] > 0 for x in present])) if present else None,
                      vein_reported_when_absent=float(np.mean([x["pred_vein_mm2"] > 0 for x in absent])) if absent else None,
                      mean_ref_vein_mm2=float(np.mean([x["ref_vein_mm2"] for x in big])) if big else None)
    vd = [v["vein_dice"] for v in per.values() if v["vein_dice"] is not None]
    med = lambda k: float(np.median([v[k] for v in per.values() if v[k] is not None]))
    summary = dict(
        subjects_scored=len(per), subjects_total=sum(len(f["held_out"]) for f in folds), folds_done=sorted(r["fold"]["fold"] for r in done),
        mean_vein_dice=float(np.mean(vd)), min_subject_vein_dice=float(np.min(vd)),
        mean_carotid_dice=float(np.mean([v["carotid_dice"] for v in per.values()])), mean_thyroid_dice=float(np.mean([v["thyroid_dice"] for v in per.values()])),
        median_area_error=med("median_area_error"), area_bias=float(np.mean([v["area_bias"] for v in per.values() if v["area_bias"] is not None])),
        artery_as_vein=float(np.mean([v["artery_as_vein"] for v in per.values()])),
        vein_found_when_present=med("vein_found_when_present"), vein_reported_when_absent=med("vein_reported_when_absent"))
    paths = sorted(p for p in FITS.glob("fold*.pt") if p.name.count(".") == 1)
    if paths:
        models = [load_model(p) for p in paths]
        a = np.random.default_rng(0).random((240, 224)).astype("float32")
        predict_probs(models, a)
        t = time.time()
        for _ in range(5):
            predict_probs(models, a)
        summary.update(seconds_per_frame=(time.time() - t) / 5, speed_device=str(DEV), speed_models=len(models))
    T = TARGETS
    checks = dict(
        mean_vein_dice=summary["mean_vein_dice"] >= T["mean_vein_dice"], min_subject_vein_dice=summary["min_subject_vein_dice"] >= T["min_subject_vein_dice"],
        median_area_error=summary["median_area_error"] <= T["median_area_error"], area_bias=abs(summary["area_bias"]) <= T["area_bias"],
        artery_as_vein=summary["artery_as_vein"] <= T["artery_as_vein"])
    if "seconds_per_frame" in summary:
        checks["seconds_per_frame"] = summary["seconds_per_frame"] <= T["seconds_per_frame"]
    report = dict(summary=summary, targets=T, targets_met=checks, per_subject=per, cfg=CFG, device=str(DEV),
                  limitation="16 healthy volunteers on Earth, reconstructed 3D slices; no astronaut, flow, clot or clinical validation. "
                             "Targets are the plan's own research bars, not clinical standards. Real-probe (NNUS/Mus-V) check not included here.")
    (RUN / "report.json").write_text(json.dumps(report, indent=1))
    pct = lambda v: "n/a" if v is None else f"{100 * v:.1f}%"
    lines = ["| Subject | Fold | Vein Dice | Carotid Dice | Thyroid Dice | Median area error | Area bias | Artery as vein |", "| --- | --- | --- | --- | --- | --- | --- | --- |"]
    for s, v in per.items():
        lines.append(f"| {s} | {v['fold']} | {v['vein_dice']:.3f} | {v['carotid_dice']:.3f} | {v['thyroid_dice']:.3f} | {pct(v['median_area_error'])} | {pct(v['area_bias'])} | {pct(v['artery_as_vein'])} |")
    (RUN / "report.md").write_text("\n".join(lines) + "\n\n" + json.dumps(dict(summary=summary, targets_met=checks), indent=1) + "\n")
    print("\n".join(lines)); print(json.dumps(dict(summary=summary, targets_met=checks), indent=1))
    return report


# ----------------------------------------------------------------------------- use on a new frame
_MODELS = None


def ensemble():
    global _MODELS
    if _MODELS is None:
        paths = sorted(p for p in FITS.glob("fold*.pt") if p.name.count(".") == 1)
        assert paths, f"no trained fold models in {FITS}"
        _MODELS = [load_model(p) for p in paths]
    return _MODELS


def predict_frame(image, spacing_mm=None):
    """image: 2D grayscale array, skin at the top. spacing_mm: mm per pixel, or None if unknown.

    With spacing the frame is resampled to the training scale and area is reported in mm2. Without it the
    frame is used as given and only scale-free outputs are returned.
    """
    from jugular_data import resample

    a = np.asarray(image, dtype="float32")
    assert a.ndim == 2 and np.isfinite(a).all()
    if spacing_mm is not None:
        a = resample(a, (max(int(round(a.shape[0] * spacing_mm / SPACING)), 8), max(int(round(a.shape[1] * spacing_mm / SPACING)), 8)), False)
    p, pm = predict_probs(ensemble(), normalise(a))
    mask = ((p + pm) / 2).argmax(0).astype("uint8")
    sp = SPACING if spacing_mm is not None else None
    m, mm = measure_frame(mask, sp), measure_frame(pm.argmax(0), sp)
    return dict(mask=mask, mask_spacing_mm=sp, measurements=m, quality=quality(m, mm))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("stages", nargs="+", choices=["cv", "report"])
    ap.add_argument("--folds", default=None, help="comma-separated fold numbers; default all")
    args = ap.parse_args()
    torch.set_num_threads(os.cpu_count() or 4)
    if "cv" in args.stages:
        stage_cv(None if args.folds is None else {int(x) for x in args.folds.split(",")})
    if "report" in args.stages:
        stage_report()

## 1. Download, audit and slice the dataset
Verifies every file's checksum, checks labels and alignment, and confirms the transverse axis two ways (header orientation and vessel roundness). It stops if the axis check fails. The scan with the undrawn vein (028_P3_1_left) is excluded.

In [ ]:
!python jugular_data.py download audit preprocess

## 2. Cross-validation by subject
Four folds. Each trains on 12 subjects and predicts the 4 held-out subjects once. Training resumes from the last finished epoch when the cells are run again, as long as the working folder survived (Kaggle: turn on file persistence in the session options; Colab: `USE_DRIVE = True`).

In [ ]:
!python -W ignore jugular_pipeline.py cv

## 3. Results
Per-subject overlap (Dice), area error against the manual outline, artery/vein confusion and speed, with the plan's acceptance targets. The targets are research bars chosen in the plan, not clinical standards.

In [ ]:
!python -W ignore jugular_pipeline.py report

In [ ]:
import json, os
import numpy as np, matplotlib.pyplot as plt
import jugular_pipeline as jp
D = jp.Data()
folds = json.loads((jp.RUN / "folds.json").read_text())
fig, axes = plt.subplots(4, 3, figsize=(11, 14))
for f, row in zip(folds, axes):
    path = sorted(p for p in jp.FITS.glob(f"fold{f['fold']}_*.pt") if p.name.count(".") == 1)[0]
    model = jp.load_model(path)
    case = D.cases[f["held_out"][0]][0]
    v = D.vols[case]; idx = np.nonzero(v["has_vein"])[0]; i = int(idx[len(idx) // 2])
    p, pm = jp.predict_probs([model], D.image(case, i))
    row[0].imshow(D.image(case, i), cmap="gray"); row[0].set_title(f"{case} (held out)")
    row[1].imshow(v["mask"][i], cmap="tab10", vmin=0, vmax=9); row[1].set_title("Manual outline")
    row[2].imshow(((p + pm) / 2).argmax(0), cmap="tab10", vmin=0, vmax=9); row[2].set_title("Prediction")
    for ax in row:
        ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Download the trained models and audit files
Four fold models (the shipped model is their average), fold assignments, per-subject table, per-slice predictions and the audit. Raw data stays under its original licence and is not included.

In [ ]:
import zipfile, os
from pathlib import Path
import jugular_pipeline as jp
ON_KAGGLE = os.path.exists("/kaggle/working")
archive = Path("/kaggle/working/jugular_results.zip" if ON_KAGGLE else "/content/jugular_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(jp.RUN.rglob("*")):
        if p.is_file() and ".last." not in p.name:
            z.write(p, p.relative_to(jp.RUN.parent))
    for name in ["audit.json", "manifest.json"]:
        z.write(jp.ROOT / name, name)
if ON_KAGGLE:
    print("Saved", archive, "- download it from the Output panel on the right")
else:
    from google.colab import files
    files.download(str(archive))